# Hadamard

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2023 Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
import torch
from IPython.display import SVG, display

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler.registers.control.multi_control import multi_control_h_ir
from flagquantum.core.ir.program.register_types import quantum_bit_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.simulation.unitary import get_unitary

## `Hadamard`
The Hadamard gate

This converts between the X and Z basis.

$$
\begin{aligned}
H |0\rangle = |+\rangle \\
H |-\rangle = |1\rangle
\end{aligned}
$$

#### Registers
 - `q`: The qubit


In [ ]:
h_signature = Signature((Register("q", quantum_bit_type()),))

### Example Instances

In [ ]:
hadamard = FieldOperation(1, multi_control_h_ir((), (), 0, (), ()))

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Hadamard", h_signature, {})])))

## `CHadamard`
The controlled Hadamard gate

#### Registers
 - `ctrl`: The control qubit.
 - `target`: The target qubit.


In [ ]:
ch_signature = Signature(tuple(Register(name, quantum_bit_type()) for name in ("ctrl", "target")))

### Example Instances

In [ ]:
chadamard = FieldOperation(2, multi_control_h_ir((0,), (1,), 1, (), ()))
assert [instruction.name for instruction in chadamard.instructions] == ["ch"]

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Controlled Hadamard", ch_signature, {})])))

In [ ]:
from io import BytesIO
from IPython.display import Image
from matplotlib import pyplot as plt
from flagquantum.drawer.register_score import RegisterStep, layout_register_score
from flagquantum.drawer.register_score_data import register_score_data
from flagquantum.drawer.register_score_mpl import draw_register_score

lanes = ("ctrl", "target")
layout = layout_register_score(
    dict.fromkeys(lanes, 1), lanes, (RegisterStep("CH", lanes, lanes, "ch"),)
)

def wire_symbol(step, register, side):
    if register is None:
        return "Text", {"text": "", "fontsize": 10}
    if step.name == "CH" and register == "ctrl":
        return "Circle", {"filled": True}
    return "TextBox", {"text": "H" if step.name == "CH" else register}

fig, _ = draw_register_score(register_score_data(layout, wire_symbol=wire_symbol))
with BytesIO() as png:
    fig.savefig(png, format="png", bbox_inches="tight")
    display(Image(data=png.getvalue()))
plt.close(fig)

### Specialty circuits

The `CHadamard` bloq is atomic and cannot be decomposed with `.decompose_bloq()`. An actual implementation on an error-corrected quantum computer will likely be architecture-dependent. A naive circuit for CHadamard can be found using Cirq.

The native circuit appends CH as one instruction. The optional teaching circuit reproduces the six leaves of the original Cirq example, with explicit matrix leaves in C, B, A application order. Its full complex matrix agrees with atomic CH. The source model declares four Cliffords and two rotations.

In [ ]:
from examples.algorithms.hadamard_tutorial import controlled_hadamard_specialty
from flagquantum.compiler._component_trace.compound_control_resource import compound_control_key
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_generalization import CliffordCall

dtype = torch.complex128
circuit = Circuit(2, dtype=dtype).append(chadamard)
reference = get_unitary(circuit)
specialty = controlled_hadamard_specialty()
expanded = Circuit(2, dtype=dtype).append(specialty)
torch.testing.assert_close(get_unitary(expanded), reference, rtol=0, atol=1e-12)
h = get_unitary(Circuit(1, dtype=dtype).append(hadamard))
torch.testing.assert_close(reference[:2, :2], torch.eye(2, dtype=dtype))
torch.testing.assert_close(reference[2:, 2:], h)
circuit.append(chadamard)
torch.testing.assert_close(get_unitary(circuit), torch.eye(4, dtype=dtype))
print(expanded.draw())
print("Specialty leaves", [gate.name for gate in specialty.instructions])
print("Source CH model", query_qec_gate_cost(compound_control_key(CliffordCall("h", 1), (1,))))